# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id`.

### Dataset Source
The dataset Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install the required `mlcroissant` library
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare for exploration. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic dataset info
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Explore available record sets and fields (`@id`s).

The dataset may have multiple record sets, each containing fields/columns. We'll list all record set `@id`s and their fields.

In [ ]:
# List all record sets with their @id and fields
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets defined in the dataset Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields & their @id:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - @id: {f.get('@id', str(f))}")
                else:
                    print(f"    - {f}")
        print("-")
    print(f"Total record sets: {len(record_sets)}")

## 3. Data Extraction
Load records from each record set using `mlcroissant`, referencing record sets by their `@id`.

We will attempt to extract data from all available record sets and place them in pandas DataFrames.

In [ ]:
dataframes = {}

# Gather record set @ids
record_set_ids = [r['@id'] for r in dataset.record_sets] if dataset.record_sets else []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Extracted {len(df)} records for record set @id: {record_set_id}")
            print("Columns:", df.columns.tolist())
        else:
            print(f"No records found for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Error reading records for record set @id: {record_set_id}: {str(e)}")

if dataframes:
    # Pick the first DataFrame for preview
    main_record_set_id = next(iter(dataframes.keys()))
    print(f"\nPreview of data from record set @id: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes extracted (no record sets with data available).")

## 4. Exploratory Data Analysis (EDA)
Let us examine the numeric fields within one record set and perform typical EDA operations: filtering, normalization, and grouping.

**Note:** Field and column references use their `@id`.

In [ ]:
import numpy as np

if not dataframes:
    print("No dataframes available for EDA.")
else:
    # Pick the first record set with data
    record_set_id = main_record_set_id
    df = dataframes[record_set_id]
    print(f"Exploring DataFrame for record set @id: {record_set_id}\nColumns: {df.columns.tolist()}")

    # Find numeric fields by checking dtypes
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Pick first numeric field @id
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # Filter above mean as arbitrary threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field
        group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < len(df) // 2 and df[col].dtype == object]
        if group_fields:
            group_field_id = group_fields[0]  # Take first likely group field by @id
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped {numeric_field_id} mean by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in this DataFrame for EDA.")

## 5. Visualization
Now let's visualize: we'll plot the distribution of a numeric field or some relation in the data for one record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set @id: {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field is available, boxplot by group
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields or dataframes available for plotting.")

## 6. Conclusion
We demonstrated how to load and inspect a Croissant-structured FAIR^2 dataset using `mlcroissant`, referencing all record sets and fields by their `@id`. This approach supports transparent, reproducible analyses on record sets, fields, and columns identified per the Croissant specification.

- Use `mlcroissant.Dataset` to load metadata and explore available record sets and fields.
- Reference all dataset entities by their `@id`.
- Extract data via `dataset.records(record_set=<record_set_id>)`.
- Process, analyze, and visualize data flexibly via pandas and matplotlib.

This notebook can be adapted for any Croissant dataset by updating the schema URL and specifying the relevant record set and field `@id`s.